In [4]:
!pip install -q transformers datasets torch scikit-learn pandas

print("All installed!")

All installed!


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
import torch
import json

print("All imports done!")
print("PyTorch version:", torch.__version__)

All imports done!
PyTorch version: 2.11.0+cpu


In [6]:
from google.colab import files

print("Please upload fever_clean.csv from your data/ folder")
uploaded = files.upload()

print("Uploaded successfully!")

Please upload fever_clean.csv from your data/ folder


Saving fever_clean.csv to fever_clean.csv
Uploaded successfully!


In [7]:
df = pd.read_csv('fever_clean.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nLabel distribution:")
print(df['our_label'].value_counts())
print("\nSample row:")
print(df.iloc[0])

Shape: (50000, 4)

Columns: ['premise', 'hypothesis', 'label', 'our_label']

Label distribution:
our_label
FACTUAL          29736
HALLUCINATION    11567
UNCERTAIN         8697
Name: count, dtype: int64

Sample row:
premise       Nikolaj Coster-Waldau worked with the Fox Broa...
hypothesis    The Fox Broadcasting Company ( often shortened...
label                                                         0
our_label                                               FACTUAL
Name: 0, dtype: object


In [8]:
# DeBERTa needs numbers not text labels
label2id = {
    'FACTUAL': 0,
    'UNCERTAIN': 1,
    'HALLUCINATION': 2
}
id2label = {0: 'FACTUAL', 1: 'UNCERTAIN', 2: 'HALLUCINATION'}

df['label_id'] = df['our_label'].map(label2id)

print("Label mapping:")
for k, v in label2id.items():
    print(f"  {k} → {v}")

print("\nLabel ID distribution:")
print(df['label_id'].value_counts())

Label mapping:
  FACTUAL → 0
  UNCERTAIN → 1
  HALLUCINATION → 2

Label ID distribution:
label_id
0    29736
2    11567
1     8697
Name: count, dtype: int64


In [9]:
# We have more FACTUAL than HALLUCINATION examples
# Undersample FACTUAL to balance the dataset

factual = df[df['our_label'] == 'FACTUAL'].sample(n=11000, random_state=42)
hallucination = df[df['our_label'] == 'HALLUCINATION']
uncertain = df[df['our_label'] == 'UNCERTAIN']

df_balanced = pd.concat([factual, hallucination, uncertain]).reset_index(drop=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("Balanced dataset size:", len(df_balanced))
print("\nBalanced label distribution:")
print(df_balanced['our_label'].value_counts())

Balanced dataset size: 31264

Balanced label distribution:
our_label
HALLUCINATION    11567
FACTUAL          11000
UNCERTAIN         8697
Name: count, dtype: int64


In [10]:
train_df, val_df = train_test_split(
    df_balanced,
    test_size=0.2,
    random_state=42,
    stratify=df_balanced['label_id']
)

print("Training set size:", len(train_df))
print("Validation set size:", len(val_df))

print("\nTrain label distribution:")
print(train_df['our_label'].value_counts())

print("\nVal label distribution:")
print(val_df['our_label'].value_counts())

Training set size: 25011
Validation set size: 6253

Train label distribution:
our_label
HALLUCINATION    9253
FACTUAL          8800
UNCERTAIN        6958
Name: count, dtype: int64

Val label distribution:
our_label
HALLUCINATION    2314
FACTUAL          2200
UNCERTAIN        1739
Name: count, dtype: int64


In [11]:
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/deberta-v3-base"
)

print("Tokenizer loaded!")
print("Vocab size:", tokenizer.vocab_size)

# Fix — convert to string to handle any NaN values
sample = df_balanced.iloc[0]
tokens = tokenizer(
    str(sample['premise']),
    str(sample['hypothesis']),
    truncation=True,
    max_length=256,
    padding='max_length',
    return_tensors='pt'
)

print("\nSample tokenization:")
print("Input IDs shape:", tokens['input_ids'].shape)
print("Attention mask shape:", tokens['attention_mask'].shape)
print("\nTokenizer working! ✅")

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenizer loaded!
Vocab size: 128000

Sample tokenization:
Input IDs shape: torch.Size([1, 256])
Attention mask shape: torch.Size([1, 256])

Tokenizer working! ✅


In [12]:
def tokenize_data(df, tokenizer, max_length=256):
    encodings = tokenizer(
        df['premise'].astype(str).tolist(),
        df['hypothesis'].astype(str).tolist(),
        truncation=True,
        max_length=max_length,
        padding='max_length',
        return_tensors='pt'
    )
    return encodings

print("Tokenizing training set... (takes 2-3 minutes)")
train_encodings = tokenize_data(train_df, tokenizer)

print("Tokenizing validation set...")
val_encodings = tokenize_data(val_df, tokenizer)

print("\nTraining encodings shape:", train_encodings['input_ids'].shape)
print("Validation encodings shape:", val_encodings['input_ids'].shape)
print("\nTokenization complete! ✅")

Tokenizing training set... (takes 2-3 minutes)
Tokenizing validation set...

Training encodings shape: torch.Size([25011, 256])
Validation encodings shape: torch.Size([6253, 256])

Tokenization complete! ✅


In [13]:
class HallucinationDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: val[idx] for key, val in self.encodings.items()
        }
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_dataset = HallucinationDataset(
    train_encodings,
    train_df['label_id'].tolist()
)
val_dataset = HallucinationDataset(
    val_encodings,
    val_df['label_id'].tolist()
)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))

sample = train_dataset[0]
print("\nSample keys:", list(sample.keys()))
print("Input IDs shape:", sample['input_ids'].shape)
print("Label:", sample['labels'])
print("\nPyTorch datasets ready! ✅")

Train dataset size: 25011
Val dataset size: 6253

Sample keys: ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
Input IDs shape: torch.Size([256])
Label: tensor(0)

PyTorch datasets ready! ✅


In [14]:
torch.save(train_dataset, 'train_dataset.pt')
torch.save(val_dataset, 'val_dataset.pt')

with open('label_mapping.json', 'w') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f)

train_df.to_csv('train_df.csv', index=False)
val_df.to_csv('val_df.csv', index=False)

print("Saved train_dataset.pt ✅")
print("Saved val_dataset.pt ✅")
print("Saved label_mapping.json ✅")
print("Saved train_df.csv ✅")
print("Saved val_df.csv ✅")

Saved train_dataset.pt ✅
Saved val_dataset.pt ✅
Saved label_mapping.json ✅
Saved train_df.csv ✅
Saved val_df.csv ✅


In [15]:
from google.colab import files

files.download('train_dataset.pt')
files.download('val_dataset.pt')
files.download('label_mapping.json')
files.download('train_df.csv')
files.download('val_df.csv')

print("All downloaded! ✅")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

All downloaded! ✅
